# 01 Bronze and Silver: load and parity

Raw capture files go to Bronze with delta-rs (append-only, one row per captured line). Silver applies the EC HW2 parser rules. This notebook shows the row counts and the parity checks against the published EC HW2 tables (`data/parity_ref/`).

Commands that made the tables: `python -m etl.run bronze --raw-dir data/raw`, `python -m etl.run silver`, `python -m etl.history all`, `python -m etl.parity bronze|silver|cohorts`.

In [1]:
import json, os, sys
from pathlib import Path
REPO = Path.cwd().resolve().parents[1] if Path.cwd().name == 'final' else Path.cwd()
sys.path.insert(0, str(REPO))
os.environ.setdefault('TRUTHLAG_HW2_DIR', str(REPO / 'ec-hw2'))
import pandas as pd
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 20)
from deltalake import DeltaTable
LAKE = REPO / 'data' / 'lake'
RUNS = LAKE / '_runs'
def run(name): return json.loads((RUNS / name).read_text())
def table(name, version=None, columns=None):
    return DeltaTable(str(LAKE / name), version=version).to_pandas(columns=columns)
def history(name, n=10):
    h = DeltaTable(str(LAKE / name)).history(n)
    return pd.DataFrame([{k: e.get(k) for k in ('version', 'operation', 'step', 'as_of', 'snapshot', 'rows')} for e in h])


## Bronze tables: versions and commits

In [2]:
for t in ['bronze/page_change_events', 'bronze/revert_risk_events', 'bronze/tags_events', 'bronze/history_slim', 'bronze/history_full']:
    print(t, 'version', DeltaTable(str(LAKE / t)).version())
history('bronze/page_change_events', 5)

bronze/page_change_events version 49
bronze/revert_risk_events version 32
bronze/tags_events version 44
bronze/history_slim version 11
bronze/history_full version 1


,version,operation,step,as_of,snapshot,rows
0,49,WRITE,bronze,None,None,71014
1,48,WRITE,bronze,None,None,100000
2,47,WRITE,bronze,None,None,100000
3,46,WRITE,bronze,None,None,100000
4,45,WRITE,bronze,None,None,100000


## Bronze parity with the published parse cut

The cut is rebuilt from `_SUCCESS_parsed.json`: hour folders up to 2026-10-27 hour 20, minus files skipped as open, cut-short files limited to the lines the parse kept.

In [3]:
bp = run('bronze_parity.json')
pd.DataFrame({s.split('.')[1] if 'revision' not in s else 'tags': {k: v for k, v in d.items() if k != 'bronze_rows_per_day'} for s, d in bp.items()}).T

,files_in_cut,files_read_published,bronze_lines_in_cut,lines_published,match,cut_short_files_that_grew_since,files_from_later_sessions,bronze_rows
page_change,232,232,1231430,1231430,True,0,0,4871014
page_revert_risk_prediction_change,230,230,789634,789634,True,0,0,3113810
tags,286,286,1603253,1603253,True,0,0,4343855


## Bronze rows per event date

In [4]:
pd.DataFrame({s.split('mediawiki.')[1]: d['bronze_rows_per_day'] for s, d in bp.items()}).fillna(0).astype(int)

,page_change.v1,page_revert_risk_prediction_change.v1,revision-tags-change
2026-10-19,5756,3079,147174
2026-10-20,161649,95059,157771
2026-10-21,128814,84462,135537
2026-10-22,137043,89493,139945
2026-10-23,158094,95507,153334
2026-10-24,155627,107418,142640
2026-10-25,183349,124679,162852
2026-10-26,162638,102816,156603
2026-10-27,161578,100288,144273
2026-10-28,185389,115247,144707


## Silver parity at the published cut (meta_dt <= 2026-10-27T20:59:59.999999Z)

In [5]:
sp = run('silver_parity.json')
pd.DataFrame({k: {c: v[c] for c in ('rows_lake', 'rows_ref', 'only_in_lake', 'only_in_ref', 'same_content')} for k, v in sp.items()}).T

,rows_lake,rows_ref,only_in_lake,only_in_ref,same_content
page_change,1231427,1231427,0,0,True
tag_events,1603253,1603253,0,0,True
revert_risk,789631,789631,0,0,True


## Cohort tables against `data/tables` (built at the parity as_of)

In [6]:
cp = run('cohort_parity.json')
pd.DataFrame({k: {c: v[c] for c in ('rows_lake', 'rows_ref', 'only_in_lake', 'only_in_ref', 'same_content', 'lake_version')} for k, v in cp.items()}).T

,rows_lake,rows_ref,only_in_lake,only_in_ref,same_content,lake_version
history_revisions,295016,295016,0,0,True,1
history_revert_links,19858,19858,0,0,True,1
live_revisions,279466,279466,0,0,True,2
live_revert_links,18358,18358,0,0,True,2
live_revert_links_all,59127,59127,0,0,True,2
late_reverts,4350,4350,0,0,True,2


## Capture gaps over 120 s

In [7]:
table('silver/capture_gaps')

,stream,topic,partition,gap_start,gap_end,gap_s
0,mediawiki.page_revert_risk_prediction_change.v1,eqiad.mediawiki.page_revert_risk_prediction_ch...,0,2026-10-21 11:38:05.506000+00:00,2026-10-21 11:41:23.588000+00:00,198.082
1,mediawiki.page_revert_risk_prediction_change.v1,eqiad.mediawiki.page_revert_risk_prediction_ch...,0,2026-10-21 11:41:23.588000+00:00,2026-10-21 11:47:11.283000+00:00,347.695
2,mediawiki.page_revert_risk_prediction_change.v1,eqiad.mediawiki.page_revert_risk_prediction_ch...,0,2026-10-26 14:08:37.818000+00:00,2026-10-26 14:13:41.814000+00:00,303.996
3,mediawiki.page_revert_risk_prediction_change.v1,codfw.mediawiki.page_revert_risk_prediction_ch...,0,2026-10-28 15:02:53.160000+00:00,2026-10-28 15:05:27.748000+00:00,154.588
4,mediawiki.revision-tags-change,codfw.mediawiki.revision-tags-change,0,2026-10-28 15:03:13.434000+00:00,2026-10-28 15:05:27.634000+00:00,134.200
